# 🎨 AnimeGANv2 미세조정 — 우리 부스만의 웹툰 그림체 모델 만들기

**무엇을 하나**: 이미 얼굴 사진→그림 변환을 아는 AnimeGANv2(`face_paint_512_v2`)를 출발점으로,
**내가 모은 그림체 이미지**로 다시 학습시켜(전이 학습) 그 그림체로 얼굴을 그리는 모델을 만든다.
끝나면 `animegan_<이름>.onnx` 파일 하나가 나오고, 부스 앱의 `models/` 폴더에 넣으면 바로 그림체 목록에 추가된다.

**준비물**
- Colab **GPU 런타임** (메뉴 → 런타임 → 런타임 유형 변경 → T4 GPU). 무료 T4로 1~2시간이면 충분.
- **그림체 이미지**: 없어도 된다! 3-b 칸에서 그림 생성 모델(Stable Diffusion, 공개 라이선스)로 "한국 웹툰풍 얼굴" 수백 장을 Colab 안에서 직접 만든다 (T4로 500장 ≈ 25분).
  직접 그린 그림이나 자유 이용 일러스트가 있으면 `style.zip`으로 올려 섞어도 된다. 웹툰 플랫폼 화면을 긁어오는 건 이용약관 위반이니 피하자.
- (선택) 사진 쪽 데이터: 기본은 공개 얼굴 데이터셋을 자동으로 받지만, 팀원 사진 200장 이상을 올리면 부스 조명에 더 잘 맞는다.

**책과 연결**: 전이 학습(7장), 데이터 준비(3·4장), 학습 모니터링(6장), 모델 서빙용 변환(8장).

In [ ]:
#@title 1) 환경 확인 + 라이브러리
import torch, subprocess, os, sys
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 → 런타임 유형을 GPU로 바꾸세요")
!pip -q install onnx onnxruntime onnxsim datasets opencv-python-headless tqdm 2>/dev/null
import cv2, numpy as np, onnx
print("torch", torch.__version__, "cv2", cv2.__version__)

## 2) 데이터 올리기

왼쪽 📁 파일 탭에 **`style.zip`**(그림체 이미지) 을 끌어다 놓거나, 구글 드라이브에 두고 아래 경로를 바꿔라.
사진 데이터도 직접 올리려면 **`photos.zip`** 으로.

In [ ]:
#@title 2) 압축 풀기 (드라이브에 있으면 경로 수정)
STYLE_ZIP = "/content/style.zip"   #@param {type:"string"}
PHOTO_ZIP = "/content/photos.zip"  #@param {type:"string"}
USE_DRIVE = False                  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
import zipfile, glob, shutil
os.makedirs("/content/data/style_raw", exist_ok=True); os.makedirs("/content/data/photo_raw", exist_ok=True)
if os.path.exists(STYLE_ZIP):
    zipfile.ZipFile(STYLE_ZIP).extractall("/content/data/style_raw")
else:
    print("style.zip 없음 → 3-b 칸에서 그림체 이미지를 생성합니다")
if os.path.exists(PHOTO_ZIP):
    zipfile.ZipFile(PHOTO_ZIP).extractall("/content/data/photo_raw")
imgs = lambda d: [p for p in glob.glob(d + "/**/*", recursive=True) if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
print("그림체 이미지:", len(imgs("/content/data/style_raw")), "장 / 내 사진:", len(imgs("/content/data/photo_raw")), "장")

In [ ]:
#@title 3) (사진을 안 올렸으면) 공개 얼굴 사진 데이터셋 받기
N_PHOTOS = 3000  #@param {type:"integer"}
if len(imgs("/content/data/photo_raw")) < 100:
    from datasets import load_dataset
    ok = False
    for name in ["huggan/CelebA-faces", "nielsr/CelebA-faces", "huggan/CelebA-HQ"]:
        try:
            ds = load_dataset(name, split="train", streaming=True)
            for i, ex in enumerate(ds):
                im = ex["image"] if "image" in ex else list(ex.values())[0]
                im.convert("RGB").save(f"/content/data/photo_raw/{i:05d}.jpg", quality=92)
                if i + 1 >= N_PHOTOS: break
            ok = True; print("사용한 데이터셋:", name); break
        except Exception as e:
            print(name, "실패:", str(e)[:80])
    if not ok:
        # 마지막 대안: AnimeGAN 원저자 배포 사진 세트 (풍경 위주라 얼굴엔 덜 맞음)
        !wget -q https://github.com/TachibanaYoshino/AnimeGAN/releases/download/dataset-1/dataset.zip -O /content/dataset.zip
        zipfile.ZipFile("/content/dataset.zip").extractall("/content/animegan_ds")
        for i, p in enumerate(glob.glob("/content/animegan_ds/dataset/train_photo/*")[:N_PHOTOS]):
            shutil.copy(p, f"/content/data/photo_raw/{i:05d}.jpg")
print("사진:", len(imgs("/content/data/photo_raw")), "장")

In [ ]:
#@title 3-b) 그림체 이미지 만들기 — 그림 생성 모델로 '한국 웹툰풍 얼굴' N장 (style.zip이 있어도 추가로 섞을 수 있음)
N_STYLE = 300            #@param {type:"integer"}
STYLE_PROMPT = "korean manhwa webtoon style, realistic proportions, natural sized eyes, korean teenager portrait, dark hair, clean thin ink lineart, soft muted colors, gentle soft shading, simple plain background, bust shot, best quality"  #@param {type:"string"}
NEG = "anime, huge eyes, sparkling eyes, purple hair, pink hair, blue hair, photo, 3d, blurry, lowres, bad anatomy, bad hands, extra fingers, text, watermark, signature, deformed, ugly, multiple people, nsfw"
if len(imgs("/content/data/style_raw")) < N_STYLE:
    !pip -q install diffusers transformers accelerate safetensors 2>/dev/null
    import torch, random
    from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
    pipe = None
    for repo in ["dreamlike-art/dreamlike-anime-1.0", "stablediffusionapi/anything-v5", "Linaqruf/anything-v3.0", "stable-diffusion-v1-5/stable-diffusion-v1-5"]:
        try:
            pipe = StableDiffusionPipeline.from_pretrained(repo, torch_dtype=torch.float16, safety_checker=None).to("cuda")
            print("생성 모델:", repo); break
        except Exception as e:
            print(repo, "실패:", str(e)[:80])
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.enable_attention_slicing()
    # 다양성: 성별·머리·표정·각도·옷을 랜덤으로 섞어 넣는다 (한 가지만 학습하면 모델이 그쪽으로 쏠림)
    who = ["teenage boy", "teenage girl", "young man", "young woman", "high school student"]
    hair = ["short black hair", "wavy brown hair", "long dark hair", "two-block haircut", "bangs covering forehead", "ponytail", "curly hair", "short hair with glasses"]
    face = ["smiling", "laughing with eyes closed", "neutral expression", "surprised", "grinning showing teeth", "winking", "shy smile", "serious look"]
    angle = ["front view", "slight three-quarter view", "looking at viewer", "head tilted"]
    cloth = ["school uniform", "hoodie", "t-shirt", "collared shirt", "sweater"]
    os.makedirs("/content/data/style_raw/gen", exist_ok=True)
    k = len(imgs("/content/data/style_raw"))
    g = torch.Generator("cuda")
    while k < N_STYLE:
        prompt = f"{STYLE_PROMPT}, {random.choice(who)}, {random.choice(hair)}, {random.choice(face)}, {random.choice(angle)}, {random.choice(cloth)}"
        seed = random.randint(0, 10**9); g.manual_seed(seed)
        ims = pipe(prompt, negative_prompt=NEG, num_inference_steps=22, guidance_scale=7.0, width=512, height=512, num_images_per_prompt=2, generator=g).images
        for im in ims:
            im.save(f"/content/data/style_raw/gen/{k:05d}.png"); k += 1
        if k % 50 < 2: print(k, "장"); display(ims[0].resize((256, 256)))
    del pipe; torch.cuda.empty_cache()
print("그림체 이미지:", len(imgs("/content/data/style_raw")), "장")

In [ ]:
#@title 3-c) 자동 저장: 만든 그림체 이미지를 zip으로 내 컴퓨터에 다운로드 (다음에 style.zip으로 올리면 생성 단계 생략)
import shutil
shutil.make_archive("/content/style", "zip", "/content/data/style_raw")
print("style.zip", round(os.path.getsize("/content/style.zip") / 1e6, 1), "MB")
from google.colab import files
files.download("/content/style.zip")

In [ ]:
#@title 4) 전처리: 크기 맞추기(256) + 그림체 '가장자리 뭉갠 판' + 흑백 판 만들기
# AnimeGAN 논문 방식: 그림체 이미지의 선 가장자리를 흐린 버전을 '가짜'로 같이 학습시키면 결과가 뿌옇지 않고 선명해진다.
import random
SIZE = 256
def prep(src_dir, dst_dir, n_max, face_crop=False):
    os.makedirs(dst_dir, exist_ok=True)
    files = imgs(src_dir); random.shuffle(files)
    k = 0
    for p in files:
        im = cv2.imread(p)
        if im is None: continue
        h, w = im.shape[:2]
        s = SIZE / min(h, w)
        if s > 1 and min(h, w) < SIZE * 0.6: continue       # 너무 작은 건 버림
        im = cv2.resize(im, (max(SIZE, int(w * s)), max(SIZE, int(h * s))), interpolation=cv2.INTER_AREA)
        h, w = im.shape[:2]
        # 가운데 + 랜덤 위치 두 장씩 잘라 데이터 2배
        for _ in range(2):
            y = random.randint(0, h - SIZE); x = random.randint(0, w - SIZE)
            cv2.imwrite(f"{dst_dir}/{k:06d}.jpg", im[y:y + SIZE, x:x + SIZE], [cv2.IMWRITE_JPEG_QUALITY, 95]); k += 1
        if k >= n_max: break
    return k
print("그림체:", prep("/content/data/style_raw", "/content/data/style", 4000))
print("사진:", prep("/content/data/photo_raw", "/content/data/photo", 6000))

def edge_smooth(img):
    # 선 가장자리만 흐리게 (논문의 edge-smoothed set)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    e = cv2.Canny(g, 100, 200)
    e = cv2.dilate(e, np.ones((5, 5), np.uint8))
    bl = cv2.GaussianBlur(img, (5, 5), 0)
    out = img.copy(); out[e > 0] = bl[e > 0]
    return out
os.makedirs("/content/data/smooth", exist_ok=True)
for p in glob.glob("/content/data/style/*.jpg"):
    cv2.imwrite(p.replace("/style/", "/smooth/"), edge_smooth(cv2.imread(p)))
print("완료. 예시 ↓")
from IPython.display import display
from PIL import Image
sample = sorted(glob.glob("/content/data/style/*.jpg"))[:4]
display(Image.fromarray(cv2.cvtColor(np.hstack([cv2.imread(p) for p in sample]), cv2.COLOR_BGR2RGB)))

In [ ]:
#@title 5) 모델 정의 (bryandlee/animegan2-pytorch 구조 그대로 → 부스 앱의 ONNX 변환과 호환)
import torch, torch.nn as nn, torch.nn.functional as F

class ConvNormLReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, pad_mode="reflect", groups=1, bias=False):
        pad_layer = {"zero": nn.ZeroPad2d, "same": nn.ReplicationPad2d, "reflect": nn.ReflectionPad2d}
        super().__init__(pad_layer[pad_mode](padding),
                         nn.Conv2d(in_ch, out_ch, kernel_size, stride, 0, groups=groups, bias=bias),
                         nn.GroupNorm(1, out_ch, affine=True), nn.LeakyReLU(0.2, inplace=True))

class InvertedResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, expansion_ratio=2):
        super().__init__()
        self.use_res_connect = in_ch == out_ch
        bottleneck = int(round(in_ch * expansion_ratio))
        layers = []
        if expansion_ratio != 1:
            layers.append(ConvNormLReLU(in_ch, bottleneck, kernel_size=1, padding=0))
        layers += [ConvNormLReLU(bottleneck, bottleneck, groups=bottleneck, bias=True),
                   nn.Conv2d(bottleneck, out_ch, 1, 1, 0, bias=False), nn.GroupNorm(1, out_ch, affine=True)]
        self.layers = nn.Sequential(*layers)
    def forward(self, x):
        out = self.layers(x)
        return out + x if self.use_res_connect else out

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.block_a = nn.Sequential(ConvNormLReLU(3, 32, 7, padding=3), ConvNormLReLU(32, 64, stride=2, padding=(0, 1, 0, 1)), ConvNormLReLU(64, 64))
        self.block_b = nn.Sequential(ConvNormLReLU(64, 128, stride=2, padding=(0, 1, 0, 1)), ConvNormLReLU(128, 128))
        self.block_c = nn.Sequential(ConvNormLReLU(128, 128), InvertedResBlock(128, 256, 2), InvertedResBlock(256, 256, 2),
                                     InvertedResBlock(256, 256, 2), InvertedResBlock(256, 256, 2), ConvNormLReLU(256, 128))
        self.block_d = nn.Sequential(ConvNormLReLU(128, 128), ConvNormLReLU(128, 128))
        self.block_e = nn.Sequential(ConvNormLReLU(128, 64), ConvNormLReLU(64, 64), ConvNormLReLU(64, 32, 7, padding=3))
        self.out_layer = nn.Sequential(nn.Conv2d(32, 3, 1, 1, 0, bias=False), nn.Tanh())
    def forward(self, x):
        out = self.block_a(x); half = self.block_b(out); out = self.block_c(half)
        out = F.interpolate(out, half.size()[-2:], mode="bilinear", align_corners=True); out = self.block_d(out)
        out = F.interpolate(out, x.size()[-2:], mode="bilinear", align_corners=True); out = self.block_e(out)
        return self.out_layer(out)

SN = nn.utils.spectral_norm
class Discriminator(nn.Module):
    def __init__(self, ch=32):
        super().__init__()
        self.net = nn.Sequential(
            SN(nn.Conv2d(3, ch, 3, 1, 1)), nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(ch, ch * 2, 3, 2, 1)), nn.LeakyReLU(0.2, True), SN(nn.Conv2d(ch * 2, ch * 4, 3, 1, 1)), nn.GroupNorm(1, ch * 4), nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(ch * 4, ch * 4, 3, 2, 1)), nn.LeakyReLU(0.2, True), SN(nn.Conv2d(ch * 4, ch * 8, 3, 1, 1)), nn.GroupNorm(1, ch * 8), nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(ch * 8, ch * 8, 3, 1, 1)), nn.GroupNorm(1, ch * 8), nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(ch * 8, 1, 3, 1, 1)))
    def forward(self, x): return self.net(x)

dev = "cuda"
G = Generator().to(dev); D = Discriminator().to(dev)
# 출발점: 얼굴 그림 모델 가중치 (전이 학습)
!wget -q https://github.com/bryandlee/animegan2-pytorch/raw/main/weights/face_paint_512_v2.pt -O /content/face_paint_512_v2.pt
G.load_state_dict(torch.load("/content/face_paint_512_v2.pt", map_location=dev))
print("G params:", sum(p.numel() for p in G.parameters()) / 1e6, "M")

In [ ]:
#@title 6) 손실 함수 (VGG 내용 보존 · 흑백 스타일 · 색 유지 · 적대적) + 데이터 로더
import torchvision
from torch.utils.data import Dataset, DataLoader
vgg = torchvision.models.vgg19(weights=torchvision.models.VGG19_Weights.IMAGENET1K_V1).features[:26].to(dev).eval()
for p in vgg.parameters(): p.requires_grad = False
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
def feats(x):  # x in [-1,1]
    return vgg(((x + 1) / 2 - MEAN) / STD)
def gram(f):
    b, c, h, w = f.shape; f = f.view(b, c, h * w); return f @ f.transpose(1, 2) / (c * h * w)
def to_gray(x):
    g = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1); return g.repeat(1, 3, 1, 1)
def to_yuv(x):
    r, g, b = x[:, 0], x[:, 1], x[:, 2]
    return torch.stack([0.299 * r + 0.587 * g + 0.114 * b, -0.147 * r - 0.289 * g + 0.436 * b, 0.615 * r - 0.515 * g - 0.100 * b], 1)
def tv_loss(x):
    return (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean() + (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()

class Folder(Dataset):
    def __init__(self, d, flip=True): self.files = sorted(glob.glob(d + "/*.jpg")); self.flip = flip
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        im = cv2.cvtColor(cv2.imread(self.files[i]), cv2.COLOR_BGR2RGB)
        if self.flip and random.random() < 0.5: im = im[:, ::-1]
        return torch.from_numpy(np.ascontiguousarray(im)).permute(2, 0, 1).float() / 127.5 - 1
BATCH = 6
photo_dl = DataLoader(Folder("/content/data/photo"), BATCH, shuffle=True, drop_last=True, num_workers=2)
style_dl = DataLoader(Folder("/content/data/style"), BATCH, shuffle=True, drop_last=True, num_workers=2)
smooth_dl = DataLoader(Folder("/content/data/smooth"), BATCH, shuffle=True, drop_last=True, num_workers=2)
print(len(photo_dl), len(style_dl), "배치/에폭")

In [ ]:
#@title 7) 학습 (T4 기준 1에폭(400스텝) ≈ 12분). 에폭마다 미리보기 + 가중치 자동 다운로드
EPOCHS = 3           #@param {type:"integer"}
STEPS_PER_EPOCH = 400  #@param {type:"integer"}
# 적대적 손실 300/300은 판별기가 너무 세서 결과가 빗금투성이로 무너짐(실험) → 30/30, 내용 보존 3.0
W_ADV_G, W_ADV_D = 30.0, 30.0
W_CON, W_STY, W_COL, W_TV = 3.0, 2.0, 10.0, 1.0   # 내용 보존 / 그림체 / 색 유지 / 매끈함 (그림체가 약하면 W_STY↑, 얼굴이 무너지면 W_CON↑)
optG = torch.optim.Adam(G.parameters(), 2e-5, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), 4e-5, betas=(0.5, 0.999))
from tqdm import tqdm
import itertools
os.makedirs("/content/ckpt", exist_ok=True); os.makedirs("/content/preview", exist_ok=True)
preview = next(iter(photo_dl))[:4].to(dev)
step = 0
for ep in range(EPOCHS):
    G.train()
    for photo, style, smooth in tqdm(zip(itertools.islice(photo_dl, STEPS_PER_EPOCH), itertools.cycle(style_dl), itertools.cycle(smooth_dl)), total=STEPS_PER_EPOCH, desc=f"epoch {ep+1}/{EPOCHS}"):
        photo, style, smooth = photo.to(dev), style.to(dev), smooth.to(dev)
        # ---- D: 진짜 그림체 ↔ (생성 결과, 흑백 그림체, 가장자리 뭉갠 그림체)
        fake = G(photo).detach()
        d_real, d_fake = D(style), D(fake)
        d_gray, d_smooth = D(to_gray(style)), D(smooth)
        lossD = W_ADV_D * (1.7 * ((d_real - 1) ** 2).mean() + 1.7 * (d_fake ** 2).mean() + 1.7 * (d_gray ** 2).mean() + 1.0 * (d_smooth ** 2).mean()) / 4
        optD.zero_grad(); lossD.backward(); optD.step()
        # ---- G
        fake = G(photo)
        adv = W_ADV_G * ((D(fake) - 1) ** 2).mean()
        f_p, f_f = feats(photo), feats(fake)
        con = W_CON * (f_p - f_f).abs().mean()
        sty = W_STY * (gram(feats(to_gray(fake))) - gram(feats(to_gray(style)))).abs().mean()
        yp, yf = to_yuv((photo + 1) / 2), to_yuv((fake + 1) / 2)
        col = W_COL * ((yp[:, 0] - yf[:, 0]).abs().mean() + F.huber_loss(yf[:, 1], yp[:, 1]) + F.huber_loss(yf[:, 2], yp[:, 2]))
        tv = W_TV * tv_loss(fake)
        lossG = adv + con + sty + col + tv
        optG.zero_grad(); lossG.backward(); optG.step()
        step += 1
        if step % 100 == 0:
            print(f"step {step}  D {lossD.item():.2f}  G {lossG.item():.2f} (adv {adv.item():.1f} con {con.item():.2f} sty {sty.item():.2f} col {col.item():.2f})")
    G.eval()
    with torch.no_grad():
        out = G(preview)
    grid = torch.cat([preview, out], 0)
    torchvision.utils.save_image((grid + 1) / 2, f"/content/preview/ep{ep+1:02d}.jpg", nrow=4)
    torch.save(G.state_dict(), f"/content/ckpt/G_ep{ep+1:02d}.pt")
    # 자동 저장: 에폭마다 가중치(8.6MB)를 내 컴퓨터 다운로드 폴더로 (세션이 끊겨도 안 날아가게)
    try:
        from google.colab import files; files.download(f"/content/ckpt/G_ep{ep+1:02d}.pt")
    except Exception as e:
        print("다운로드 실패(수동으로 받으세요):", e)
    display(Image.open(f"/content/preview/ep{ep+1:02d}.jpg"))
print("학습 끝. /content/ckpt 에 에폭별 가중치, /content/preview 에 미리보기")

## 8) 결과 고르기
위에 에폭마다 뜬 미리보기(위 줄: 사진, 아래 줄: 결과)를 보고 **가장 마음에 드는 에폭**을 고른다.
- 얼굴이 무너지거나 눈이 이상해지면 → 더 이른 에폭을 고르거나, `W_CON`을 2.0~3.0으로 올려 다시
- 그림체가 너무 약하면(사진 같으면) → `W_STY`를 4~6으로, 에폭을 더

In [ ]:
#@title 9) 테스트: 내 사진 한 장으로 512 해상도 확인 (test.jpg 를 올려두고 실행)
BEST_EPOCH = 3    #@param {type:"integer"}
G.load_state_dict(torch.load(f"/content/ckpt/G_ep{BEST_EPOCH:02d}.pt", map_location=dev)); G.eval()
p = "/content/test.jpg"
if os.path.exists(p):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); im = cv2.resize(im, (512, 512))
    x = torch.from_numpy(im).permute(2, 0, 1).float().unsqueeze(0).to(dev) / 127.5 - 1
    with torch.no_grad(): y = G(x)
    out = ((y[0].permute(1, 2, 0).cpu().numpy() + 1) * 127.5).clip(0, 255).astype(np.uint8)
    display(Image.fromarray(np.hstack([im, out])))
else:
    print("test.jpg 가 없어요 — 건너뜀")

In [ ]:
#@title 10) ONNX로 내보내기 (부스 앱 형식: 입력 이름 input, 1×3×512×512, 출력 -1~1)
STYLE_NAME = "webtoon"   #@param {type:"string"}
G.eval().cpu()
dummy = torch.randn(1, 3, 512, 512)
path = f"/content/animegan_{STYLE_NAME}.onnx"
torch.onnx.export(G, dummy, path, input_names=["input"], output_names=["output"], opset_version=13, do_constant_folding=True)
try:
    import onnxsim
    m, ok = onnxsim.simplify(onnx.load(path)); onnx.save(m, path) if ok else None
except Exception as e:
    print("simplify 생략:", e)
import onnxruntime as ort
s = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
y = s.run(None, {"input": dummy.numpy()})[0]
print("OK", y.shape, y.min(), y.max(), "|", os.path.getsize(path) / 1e6, "MB")
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy(path, "/content/drive/MyDrive/"); print("드라이브에도 저장:", "/content/drive/MyDrive/" + os.path.basename(path))
from google.colab import files
files.download(path)

## 11) 부스 앱에 넣기
1. 받은 `animegan_webtoon.onnx` 를 `caricature-booth/models/` 에 복사
2. `server/pipeline/config.py` 의 `STYLES` 에 한 줄 추가:
   ```python
   ("webtoon_ft", "animegan_webtoon", "우리 웹툰풍", "우리가 모은 그림체로 직접 학습시킨 모델 (Colab, 전이 학습)"),
   ```
3. 감성 웹툰풍(하이브리드)의 바탕 채색도 이 모델로 바꾸고 싶으면 `server/pipeline/vector.py` 의 `VARIANTS["romance"]` 에서 `base="paint"` → `base="webtoon_ft"`
4. `start.bat` 다시 실행

**발표 때 설명**: "얼굴 사진→그림을 이미 배운 모델(8.6MB)을 우리 그림체 1000장으로 20에폭 더 가르쳤다(전이 학습).
생성기는 '사진 내용은 유지하되(VGG 특징) 그림체는 따라가라(흑백 질감, 판별기)'는 네 가지 손실을 동시에 줄이며 학습한다."